In [2]:
import pandas as pd, numpy as np

folder = r"D:\retailpulse\datarawsynthetic\retail_contaminated_dataset"
inv   = pd.read_csv(folder + r"\inventory_snapshot.csv")
flags = pd.read_csv(folder + r"\sku_inventory_flags.csv")[["sku_id", "flag"]]

s = pd.read_parquet(r"D:\retailpulse\retailpulse_sales_clean.parquet",
                    columns=["date", "store_id", "sku_id", "quantity"])
end = s["date"].max()
last90 = s[s["date"] > end - pd.Timedelta(days=90)]          # the last 90 days of sales

demand = (last90.groupby(["store_id", "sku_id"])["quantity"].sum() / 90).rename("daily_demand").reset_index()

inv = inv.merge(demand, on=["store_id", "sku_id"], how="left").merge(flags, on="sku_id", how="left")
inv["daily_demand"] = inv["daily_demand"].fillna(0)
inv["days_of_cover"] = inv["stock_on_hand"] / inv["daily_demand"].replace(0, np.nan)   # how many days the stock will last
inv["flag"] = inv["flag"].fillna("NORMAL")

print(inv.shape)
print(inv.groupby("flag").size())
print(inv.groupby("flag")[["stock_on_hand", "daily_demand", "days_of_cover"]].median().round(2))


(26408, 9)
flag
NORMAL           11442
SLOW_MOVER       10674
STOCKOUT_RISK     4292
dtype: int64
               stock_on_hand  daily_demand  days_of_cover
flag                                                     
NORMAL                 104.0          0.06         1575.0
SLOW_MOVER             260.0          0.03         6570.0
STOCKOUT_RISK            0.0          0.39            0.0


In [3]:
LEAD_TIME = 7     # days a new order takes to arrive (our assumption)
Z = 1.65          # 95% service level

# 1. how much demand varies from day to day (days with no sale count as 0)
daily = last90.groupby(["store_id", "sku_id", "date"])["quantity"].sum()
sumsq = (daily ** 2).groupby(level=["store_id", "sku_id"]).sum()
mean_d = demand.set_index(["store_id", "sku_id"])["daily_demand"]
std_d = ((sumsq / 90 - mean_d ** 2).clip(lower=0) ** 0.5).rename("demand_std").reset_index()
inv = inv.merge(std_d, on=["store_id", "sku_id"], how="left")
inv["demand_std"] = inv["demand_std"].fillna(0)

# 2. use the Prophet forecast: next 30 days vs the last 90 days
fc = pd.read_csv(r"D:\retailpulse\app_data\prophet_forecast.csv", parse_dates=["ds"])
sales_d = pd.read_csv(r"D:\retailpulse\app_data\daily_sales.csv", parse_dates=["date"])
factor = fc[fc["ds"] > end].head(30)["yhat"].mean() / sales_d[sales_d["date"] > end - pd.Timedelta(days=90)]["sales"].mean()
print("forecast factor:", round(factor, 2))

# 3. the inventory numbers
inv["fc_daily_demand"]    = inv["daily_demand"] * factor
inv["safety_stock_calc"]  = np.ceil(Z * inv["demand_std"] * np.sqrt(LEAD_TIME))
inv["reorder_point_calc"] = np.ceil(inv["fc_daily_demand"] * LEAD_TIME + inv["safety_stock_calc"])
inv["order_qty"]          = np.ceil(np.maximum(0, inv["fc_daily_demand"] * 30 + inv["safety_stock_calc"] - inv["stock_on_hand"]))
inv["needs_reorder"]      = inv["stock_on_hand"] <= inv["reorder_point_calc"]

print(pd.crosstab(inv["flag"], inv["needs_reorder"]))
print(inv.groupby("flag")["order_qty"].agg(["sum", "median"]))
print(inv.groupby("flag")["days_of_cover"].describe(percentiles=[.1, .25, .5, .75, .9]).round(0))

forecast factor: 0.77
needs_reorder  False  True 
flag                       
NORMAL         11201    241
SLOW_MOVER     10673      1
STOCKOUT_RISK    113   4179
                   sum  median
flag                          
NORMAL          1012.0     0.0
SLOW_MOVER         0.0     0.0
STOCKOUT_RISK  91022.0    14.0
                 count    mean     std    min     10%     25%     50%  \
flag                                                                    
NORMAL         10244.0  3662.0  5763.0    0.0   207.0   588.0  1575.0   
SLOW_MOVER      8710.0  9824.0  9151.0  120.0  2196.0  3657.0  6570.0   
STOCKOUT_RISK   4292.0     6.0    60.0    0.0     0.0     0.0     0.0   

                   75%      90%      max  
flag                                      
NORMAL          4050.0   9450.0  53280.0  
SLOW_MOVER     12510.0  22680.0  52740.0  
STOCKOUT_RISK      0.0      0.0   2597.0  


In [4]:
# stock above what the next 90 days of forecast demand (plus buffer) would need
inv["target_stock"] = np.ceil(inv["fc_daily_demand"] * 90 + inv["safety_stock_calc"])
inv["excess_units"] = np.maximum(0, inv["stock_on_hand"] - inv["target_stock"])

sku = pd.read_csv(folder + r"\sku_master.csv")[["sku_id", "sku_name", "category", "cost_price"]]
out = inv.merge(sku, on="sku_id", how="left")
out["order_cost"] = out["order_qty"] * out["cost_price"]

cols = ["store_id", "sku_id", "sku_name", "category", "flag", "stock_on_hand", "daily_demand",
        "fc_daily_demand", "safety_stock_calc", "reorder_point_calc", "needs_reorder",
        "order_qty", "order_cost", "excess_units"]
out[cols].round(2).to_csv(r"D:\retailpulse\app_data\inventory_actions.csv", index=False)

print("Total order cost:", round(out["order_cost"].sum()))
print("Excess units, slow movers vs others:")
print(out.groupby("flag")["excess_units"].sum())

Total order cost: 38987182
Excess units, slow movers vs others:
flag
NORMAL           1431749.0
SLOW_MOVER       2818429.0
STOCKOUT_RISK       5662.0
Name: excess_units, dtype: float64


In [5]:
# only order when stock has fallen to the reorder point
inv["order_qty"] = np.where(inv["needs_reorder"], inv["order_qty"], 0)
print(inv.groupby("flag")["order_qty"].agg(["sum", "median"]))

                   sum  median
flag                          
NORMAL           923.0     0.0
SLOW_MOVER         0.0     0.0
STOCKOUT_RISK  90251.0    14.0


In [6]:
   inv["needs_reorder"]      = inv["stock_on_hand"] <= inv["reorder_point_calc"]

In [7]:
   inv["order_qty"] = np.where(inv["needs_reorder"], inv["order_qty"], 0)   # only order when stock has reached the reorder point

In [8]:
import pandas as pd

path = r"D:\retailpulse\app_data\inventory_actions.csv"
fix = pd.read_csv(path)

# no order when stock has not reached the reorder point
fix.loc[~fix["needs_reorder"], ["order_qty", "order_cost"]] = 0
fix.to_csv(path, index=False)

print("rows still wrong:", ((fix["order_qty"] > 0) & ~fix["needs_reorder"]).sum())
print("Total order cost:", round(fix["order_cost"].sum()), "| units:", int(fix["order_qty"].sum()))

rows still wrong: 0
Total order cost: 38478420 | units: 91174
